## Real Data Examples

In [1]:
import os
import sys
import torch
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import argparse
import torch.nn as nn
import torch.optim as optim
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.spatial.distance import cdist, pdist
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle
import traceback
from wgf_minimal import sample_wgf_new
from hyperimpute.plugins.imputers import Imputers
import scipy

sys.path.append(str(Path("MIRI-Imputation").resolve()))
from src.imputer_wrapper import impute_now

## Enable usage of R methods
os.environ['RENV_CONFIG_AUTOLOAD_ENABLED'] = 'FALSE'
os.environ['R_PROFILE_USER'] = ''
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

ModuleNotFoundError: No module named 'wgf_minimal'

### Set parameters and load existing results if available

In [ ]:
### SETTINGS ###

datasets = ["gas", "pumadyn32nm", "scm1d", "scm20d", "parkinsons"]
methods = ['truth', 'wgf_new', 'true_impute', 'mice_cart', "mice_rf", "missForest", 'miri']

## Resampling params (still relevant)
param_vals = {
    "init": "ColBT",
    "T": 100
}

In [ ]:
# Define the log file path
save_dir = 'results'
log_file = os.path.join(save_dir, 'parameter_log.csv')

# Check if the log file exists and read it
if os.path.exists(log_file):
    param_log = pd.read_csv(log_file)
else:
    param_log = pd.DataFrame(columns=['ID', 'dataset', 'method', 'init', 'T'])

# Determine the next ID
next_id = 1 if param_log.empty else param_log['ID'].max() + 1

# Initialize results storage
results = {"data": {}, "metrics": {}, "Xhat_store": {}}

for dataset in datasets:
    results["data"][dataset] = {}
    results["metrics"][dataset] = {}
    results["Xhat_store"][dataset] = {}
    
    for method in methods:
        # Check if this (dataset, method) combination already exists
        matching = param_log.query(
            f"dataset == {repr(dataset)} & method == {repr(method)} & "
            f"init == {repr(param_vals['init'])} & T == {repr(param_vals['T'])}"
        )

        if not matching.empty:
            print(f"[SKIP] {dataset} / {method} already run (ID={matching['ID'].iloc[0]})")
            # Optionally load existing result here
            continue

        print(f"[RUN]  {dataset} / {method}")

        # --- your method logic here ---
        # e.g., Xhat = run_method(method, X_obs, ...)

        # Log this combination
        new_row = {'ID': next_id, 'dataset': dataset, 'method': method, **param_vals}
        param_log = pd.concat([param_log, pd.DataFrame([new_row])], ignore_index=True)
        next_id += 1

# Save updated log
os.makedirs(save_dir, exist_ok=True)
param_log.to_csv(log_file, index=False)


##Continue here!!

### Generate simulated dataset (if not available)

### Run sample methods and compute metrics (only those that are not available yet) 

In [ ]:
for method in methods:
    if method in results["metrics"] and len(results["metrics"][method]) > 0:
        print(f"{method} already done — skipping")
        continue

    print(f"Running {method}")
    results["metrics"][method] = []
    
    for run, data in results["data"].items():
        print(f"Run number {run+1}")
        
        torch.manual_seed(run + param_vals["n_runs"])
        np.random.seed(run + param_vals["n_runs"])
        
        if method == "truth":
            Xhats = [sample_truth(param_vals["n_new"], param_vals["d"])]
        # elif method == "wgf":
        #     Xhats = sample_wgf(data["X0"], data["M"], T=param_vals["T"], sgd_steps=1000)
        elif method == "miri":
        	Xhats = [impute_now(data["X0"], data["M"], data["Xstar"], "miri", max_rounds=15, batchsize=500, maxepochs=900, odesteps=100)[0]]
        elif method == "wgf_new":
            Xhats = sample_wgf_new(data["X0"], data["X0"], data["M"], T=param_vals["T"])
        elif method == "mice_cart" or method == "mice_rf" or method == "missForest":
            ro.r("set.seed(123)")

            # Import R package
            mice = importr("mice")
            missForest= importr("missForest")
            
            # Convert torch -> pandas
            X_miss = data["X0"].clone()
            X_miss[data["M"] == 0] = torch.nan
            X_np = X_miss.detach().cpu().numpy()
            X_df = pd.DataFrame(X_np, columns=[f"x{i}" for i in range(X_np.shape[1])])
            
            # Convert pandas -> R
            with localconverter(ro.default_converter + pandas2ri.converter):
                r_df = ro.conversion.py2rpy(X_df)
            
            # Run mice in R
            if method == "mice_cart"
                res = mice.mice(r_df, method="cart", m=1, eps=0)

            if method == "mice_rf"
                res = mice.mice(r_df, method="rf", m=1, eps=0)

            if method == "missForest"
                res = mice.missForest(r_dfs)
            
            # Call complete() from R namespace
            complete = ro.r["complete"]
            completed = complete(res, 1)
            
            # Convert R → pandas
            with localconverter(ro.default_converter + pandas2ri.converter):
                completed_df = ro.conversion.rpy2py(completed)
            
            X_imputed = torch.tensor(completed_df.values, dtype=X_miss.dtype)            
            Xhats = [X_imputed]
        else:
            raise NotImplementedError("Only methods 'truth' and 'wgf' implemented so far.")
            # TODO: Change this to allow for n_imputations > 1 and to save results of all iterations
            # Xhat = impute_now(data["X0"], data["M"], data["Xstar"], method, max_rounds=param_vals["max_rounds"])[0]
            # Xhats = [Xhat]
        

        XX_star_mean = cdist(data["Xstar"], data["Xstar"]).mean()
        for i, Xhat in enumerate(Xhats):
            Xid = f"{method}_{run}_{i}"
            results["Xhat_store"][Xid] = Xhat
            
            results["metrics"][method].append({
                "run": run,
                "iter": i,
                "Xhat_id": Xid,
                "quantile": np.quantile(Xhat, 0.1),
                "energy": energy_distance_fixed_X(data["Xstar"], XX_star_mean, Xhat)
            })

truth already done — skipping
wgf_new already done — skipping
true_impute already done — skipping
mice_cart already done — skipping
Running miri
Run number 1
Output shape: torch.Size([100, 3072])
Output values: tensor([[-4.7113e-01,  4.5210e+00,  3.2096e+00,  ..., -3.4703e-01,
         -2.8781e-01, -2.2578e-01],
        [-3.0531e-01,  1.0261e+00,  1.2741e+00,  ..., -2.7000e-01,
          1.1264e-01, -1.0890e-01],
        [ 2.2239e-01,  1.7841e+00, -6.0394e-01,  ..., -9.1411e-01,
          2.0542e-02, -1.5446e-01],
        ...,
        [ 8.1945e-02,  1.8961e+00,  2.2523e+00,  ...,  3.3891e-02,
          2.1582e-03, -9.3768e-01],
        [ 1.5207e-02,  1.8640e+00,  6.2815e-01,  ..., -1.6863e-01,
          2.8294e-02, -1.9412e-02],
        [ 2.3057e-01, -6.2004e-01,  2.3797e-02,  ..., -2.4015e+00,
         -2.7314e-01, -6.2340e-01]], grad_fn=<MulBackward0>)
Output shape: torch.Size([100, 1024])
Output values: tensor([[-2.2311e-01,  1.8444e+00, -6.5543e-01,  ...,  1.3032e-01,
          1.6

100%|██████████| 100/100 [00:32<00:00,  3.04it/s]


mi: 0.1302134394645691
Training vector field ...


 10%|▉         | 86/900 [00:59<07:49,  1.73it/s]

### Save the results

In [33]:
torch.save(results, results_file)

# Save the updated log file
param_log.to_csv(log_file, index=False)

### Plot the new complete samples

## Boxplots for RMSE, Energy and Quantile

In [36]:
## Create dataframe with all results
rows = []
for method, recs in results["metrics"].items():
    for r in recs:
        rows.append({**r, "method": method})

results_df = pd.DataFrame(rows)